# DATA GEN

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


# Generating questions with structured output

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

# The instructions for the LLM:

In [12]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

Call the LLM for one document:

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
model="llama-3.3-70b-versatile"

In [8]:
import json

user_prompt = json.dumps(doc)

In [13]:
messages = [
    {
        "role": "system",
        "content": data_gen_instructions + """

Return ONLY valid JSON.

Format:

{
  "questions": [
    "...",
    "...",
    "...",
    "...",
    "..."
  ]
}

Do not return markdown.
Do not use ```json.
Do not add explanations.
"""
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

Call the model:

In [14]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    temperature=0
)

In [15]:
result = response.choices[0].message.content

print(result)

{
  "questions": [
    "How late can I join the course and still get a certificate?",
    "Is it too late to start the course now?",
    "Can I still enroll in the course if it's already started?",
    "What's the deadline for joining the course to receive a certificate?",
    "Will I be able to get a certificate if I join the course after it's started?"
  ]
}


In [16]:
questions = json.loads(result)

print(questions)

{'questions': ['How late can I join the course and still get a certificate?', 'Is it too late to start the course now?', "Can I still enroll in the course if it's already started?", "What's the deadline for joining the course to receive a certificate?", "Will I be able to get a certificate if I join the course after it's started?"]}


In [17]:
print(questions["questions"])

['How late can I join the course and still get a certificate?', 'Is it too late to start the course now?', "Can I still enroll in the course if it's already started?", "What's the deadline for joining the course to receive a certificate?", "Will I be able to get a certificate if I join the course after it's started?"]


In [18]:
for q in questions["questions"]:
    print(q)

How late can I join the course and still get a certificate?
Is it too late to start the course now?
Can I still enroll in the course if it's already started?
What's the deadline for joining the course to receive a certificate?
Will I be able to get a certificate if I join the course after it's started?


# Reusable utilities

In [19]:
from evaluation_utils import llm_structured

In [20]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt
)

print(result["questions"])

[{'id': 1, 'text': 'Is it too late to enroll in the course if I just found out about it now?'}, {'id': 2, 'text': 'Can I still participate in the course if the start date has already passed?'}, {'id': 3, 'text': 'How do I know if I can still join the course and get a certificate?'}, {'id': 4, 'text': 'What are the requirements to join the course after the initial start date?'}, {'id': 5, 'text': 'Will I be able to get a certificate if I join the course late and complete all the work?'}]


Tracking cost

In [23]:
usage.prompt_tokens
usage.completion_tokens
usage.total_tokens

362

In [24]:
print("Input tokens:", usage.prompt_tokens)
print("Output tokens:", usage.completion_tokens)
print("Total tokens:", usage.total_tokens)

Input tokens: 213
Output tokens: 149
Total tokens: 362


Calculate the cost of this call:

In [25]:
from evaluation_utils import calc_price

cost = calc_price(usage)

print(cost)

{'input_cost': 0.0, 'output_cost': 0.0, 'total_cost': 0.0}


Now convert these questions into ground truth records:

In [26]:
records = []

for q in result["questions"]:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': {'id': 1,
   'text': 'Is it too late to enroll in the course if I just found out about it now?'},
  'document': '74eb249bbf'},
 {'question': {'id': 2,
   'text': 'Can I still participate in the course if the start date has already passed?'},
  'document': '74eb249bbf'},
 {'question': {'id': 3,
   'text': 'How do I know if I can still join the course and get a certificate?'},
  'document': '74eb249bbf'},
 {'question': {'id': 4,
   'text': 'What are the requirements to join the course after the initial start date?'},
  'document': '74eb249bbf'},
 {'question': {'id': 5,
   'text': 'Will I be able to get a certificate if I join the course late and complete all the work?'},
  'document': '74eb249bbf'}]